# Showing AMR slices with native interface

This notebook shows various examples of visualizing planes in AMR runs, with or without outlining patch boundaries.

In [1]:
import numpy as np
import matplotlib.pyplot as pl
import dispatch
import dispatch.graphics as dgr

In [2]:
import os
home='C:/cygwin64/home/codes/dispatch2/'
dir='experiments/ramses_snapshots/notebooks/'
os.chdir(home+dir)

Check if a patch contains a z-plane

In [3]:
def contains(p,z):
    return p.level >= 6 and abs(z-p.position[2]) <= p.size[2]/2

Interpolate in the log to a specific z-plane

In [46]:
def interpolate(p,iv,z,log=False):
    n=p.n[2]
    dz=p.size[2]/n
    z0=p.position[2]-p.size[2]/2+dz/2
    w=(z-z0)/dz
    i=int(w)
    i=max(0,min(i,n-2))
    w=w-i
    f=p.var(iv)
    if iv==0 or iv==4 or iv=='d' or iv=='e':
        log=True
    if log:
        f=np.log(f)
    f=(1-w)*f[:,:,i]+w*f[:,:,i+1]
    if log:
        f=np.exp(f)
    return f

Show outline

In [5]:
def outline(e):
    x=[e[0],e[0],e[1],e[1],e[0]]
    y=[e[2],e[3],e[3],e[2],e[2]]
    plot(x,y,color='grey')

Interpolation test

In [87]:
p=sn.patches[-1]
d=interpolate(p,0,p.position[2])
pl.clf()
dgr.imshow(d)

Show one z-plane

In [164]:
def plane(pp,z,iv='d',log=False,mesh=False,ident=False,verbose=0):
    pp=[p for p in pp if contains(p,z)]
    vmin=1e20
    vmax=0.0
    for p in pp:
        d=interpolate(p,iv,z)
        vmin=min(vmin,d.min())
        vmax=max(vmax,d.max())
        if verbose>0:
            print(p.id,d.min(),d.max())
    pl.clf()
    pl.xlim(0,1); pl.ylim(0,1)
    if log:
        vmin=np.log10(vmin)
        vmax=np.log10(vmax)
    for p in pp:
        e=p.extent[2]
        d=interpolate(p,iv,z)
        if log:
            d=np.log10(interpolate(p,iv,z))
        pl.imshow(d.transpose(),extent=e,
              vmin=vmin,vmax=vmax,origin='lower')
        x0,x1,y0,y1=p.extent[2]
        if mesh:
            outline(p.extent[2])
        if ident:
            x0,x1,y0,y1=p.extent[2]
            pl.text(x0,y0,'{}'.format(p.id))
    pl.colorbar()
    pl.title('z={:.2f}, t={:.4f}'.format(z,sn.time));

Loop over all distinct patch midplanes

In [12]:
sn=dispatch.snapshot(2,data='../data')
zz=[p.position[2] for p in sn.patches]
zz=np.unique(np.sort(zz))
for z in zz:
    plane(sn.patches,z,log=0)
    dgr.pause(1)

In [32]:
p=sn.patches[6]
f=p.var(7,all=1)
i=0
pl.clf()
for j in range(4):
    for k in range(4):
        pl.subplot(4,4,i+1)
        dgr.imshow(f[:,:,i+2])
        i+=1
pl.tight_layout()

In [17]:
sn=dispatch.snapshot(2,data='../data')
zz=p.position[2]-p.size[2]/2+p.ds[2]*(0.5+np.arange(16))
for z in zz:
    plane(sn.patches,z,log=0)
    dgr.pause(1)

One specific plane

In [205]:
sn=dispatch.snapshot(5,run='gmc2',data='../data/')
plane(sn.patches,z=0.8,iv=5,mesh=1,ident=1)
pl.xlim(0,0.3); pl.ylim(0.5,0.8);

Loop over time

In [176]:
for io in range(10):
    sn=dispatch.snapshot(io,data='../data')
    pl.clf()
    plane(sn.patches,z=0.85,iv=5,mesh=1)
    dgr.pause(1)

One specific patch

In [172]:
sn=dispatch.snapshot(2,data='../data')
p=sn.patches[45]
pl.clf()
d=p.var('px',all=1)[:,:,8]
dgr.imshow(d)
pl.title('id = {}  z = {:.2f}  t = {:.6f}'.format(
    p.id,p.position[2],p.time));

Seep in z

In [174]:
sn=dispatch.snapshot(2,data='../data')
p=sn.patches[44]
for i in range(22):
    pl.clf()
    d=p.var('px',all=1)[:,:,i]
    dgr.imshow(d)
    pl.title('id = {}  iz = {}  t = {:.6f}'.format(
        p.id,i,p.time));
    dgr.pause(1)